<a href="https://colab.research.google.com/github/AshishKhadka1/MachineLearning/blob/master/TroubleShootingAgent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [2]:
import pandas as pd
import numpy as np

# Set random seed for reproducibility
np.random.seed(42)

# Generate synthetic data
n_samples = 1000
data = {
    'timestamp': pd.date_range(start='2024-01-01', periods=n_samples, freq='h'),
    'cpu_usage': np.random.normal(50, 10, n_samples),       # CPU usage in percentage
    'memory_usage': np.random.normal(60, 15, n_samples),    # Memory usage in percentage
    'network_latency': np.random.normal(100, 20, n_samples), # Network latency in ms
    'disk_io': np.random.normal(75, 10, n_samples),         # Disk I/O in MB/s
    'error_rate': np.random.choice([0, 1], n_samples, p=[0.95, 0.05])  # 5% error rate
}

# Create DataFrame
df = pd.DataFrame(data)

# Display the first few rows of the dataset
print(df.head())
print(df.info())

            timestamp  cpu_usage  memory_usage  network_latency    disk_io  \
0 2024-01-01 00:00:00  54.967142     80.990332        86.496435  55.921924   
1 2024-01-01 01:00:00  48.617357     73.869505        97.109627  66.396150   
2 2024-01-01 02:00:00  56.476885     60.894456        84.151602  70.863945   
3 2024-01-01 03:00:00  65.230299     50.295948        93.840769  93.876877   
4 2024-01-01 04:00:00  47.658466     70.473350        62.127707  80.565531   

   error_rate  
0           0  
1           0  
2           1  
3           0  
4           0  
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 6 columns):
 #   Column           Non-Null Count  Dtype         
---  ------           --------------  -----         
 0   timestamp        1000 non-null   datetime64[ns]
 1   cpu_usage        1000 non-null   float64       
 2   memory_usage     1000 non-null   float64       
 3   network_latency  1000 non-null   float64       
 4   disk_io

In [3]:
from sklearn.ensemble import IsolationForest

# Implement anomaly detection using Isolation Forest
def detect_anomalies(data):
    model = IsolationForest(contamination=0.05, random_state=42)
    model.fit(data)
    anomalies = model.predict(data)
    return anomalies

# Detect anomalies in the dataset
numeric_data = df.select_dtypes(include=[float, int]) # Only numeric columns
df['anomaly'] = detect_anomalies(numeric_data)

print(df['anomaly'].value_counts()) # -1 denotes an anomaly

anomaly
 1    950
-1     50
Name: count, dtype: int64


In [4]:
from scipy.stats import zscore

# Calculate z-scores to identify anomalous values per column in anomalous rows
z_scores = numeric_data.apply(zscore)

# Function to identify anomalous columns for each row
def find_anomalous_columns(row, threshold=3):
    return [col for col in numeric_data.columns if abs(z_scores.loc[row.name, col]) > threshold]

# Apply the function to each anomalous row
df['anomalous_columns'] = df.apply(lambda row: find_anomalous_columns(row) if row['anomaly'] == -1 else [], axis=1)

# Display rows with anomalies and their anomalous columns
print(df[df['anomaly'] == -1][['timestamp', 'anomaly', 'anomalous_columns']])

              timestamp  anomaly              anomalous_columns
37  2024-01-02 13:00:00       -1                   [error_rate]
38  2024-01-02 14:00:00       -1                   [error_rate]
62  2024-01-03 14:00:00       -1                   [error_rate]
132 2024-01-06 12:00:00       -1                   [error_rate]
179 2024-01-08 11:00:00       -1                   [error_rate]
192 2024-01-09 00:00:00       -1                   [error_rate]
208 2024-01-09 16:00:00       -1                   [error_rate]
241 2024-01-11 01:00:00       -1                   [error_rate]
245 2024-01-11 05:00:00       -1                   [error_rate]
251 2024-01-11 11:00:00       -1                   [error_rate]
262 2024-01-11 22:00:00       -1        [cpu_usage, error_rate]
272 2024-01-12 08:00:00       -1                   [error_rate]
285 2024-01-12 21:00:00       -1                   [error_rate]
315 2024-01-14 03:00:00       -1                   [error_rate]
329 2024-01-14 17:00:00       -1        

In [7]:
from sklearn.tree import DecisionTreeClassifier
import pandas as pd # Ensure pandas is imported if not already globally available

# Train a decision tree for root cause analysis
# X_train: Features (numerical only), y_train: Target (anomaly labels)
# X_test: Features for prediction (numerical only)
def root_cause_analysis(X_train, y_train, X_test):
    model = DecisionTreeClassifier(random_state=42) # Added random_state for reproducibility
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)
    return predictions

# Prepare training data for Root Cause Analysis (RCA).
# Exclude 'timestamp' (datetime), 'anomaly' (the target variable itself), and 'anomalous_columns' (non-numeric, derived).
# 'errors='ignore'' handles cases where 'anomalous_columns' might not exist yet during the first run.
X_train_rca = df.drop(columns=['timestamp', 'anomaly', 'anomalous_columns'], errors='ignore')
y_train_rca = df['anomaly']

# Perform root cause analysis on the training data itself as a demonstration.
# In a real-world scenario, you would typically use a separate validation/test set.
predicted_anomalies_train = root_cause_analysis(X_train_rca, y_train_rca, X_train_rca)

print("Decision Tree predictions on training data (anomaly counts):")
print(pd.Series(predicted_anomalies_train).value_counts())

Decision Tree predictions on training data (anomaly counts):
 1    950
-1     50
Name: count, dtype: int64


In [9]:
# The 'recommend_solution' function has been moved to the troubleshooting agent cell (JfUX-pe4U0e_)
# to ensure it's always available when the agent is run. This cell now serves as a placeholder
# or for any future independent solution recommendation examples if needed.

# If you wish to test a recommendation separately, you would typically call the function
# after the agent cell (JfUX-pe4U0e_) has been executed.
# Example:
# if 'recommend_solution' in globals(): # Check if the function is defined in global scope
#     print(recommend_solution("network_error"))
# else:
#     print("The recommend_solution function is not available in the global scope.")

In [10]:
# Example solution recommendation based on root cause - now defined within this cell
# to ensure it's always available for the troubleshooting agent.
def recommend_solution(root_cause):
    solutions = {
        "network_error": "Restart the network service, check network configuration, or contact network team.",
        "database_issue": "Check the database connection, query performance, and restart the service if necessary.",
        "high_cpu_usage": "Identify and optimize CPU-intensive processes or consider scaling up resources.",
        "disk_issue": "Check disk I/O, storage capacity, and integrity. Consider optimizing I/O operations or upgrading storage.",
        "high_memory_usage": "Identify and optimize memory-intensive applications or allocate more RAM to the system.",
        "application_error": "Review application logs for specific error messages and apply relevant patches, configuration changes, or restarts."
    }
    return solutions.get(root_cause, "No specific recommendation available. Further investigation needed.")

# Simulate a network error and high CPU usage by altering the dataset for a troubleshooting scenario.
# Create a copy of the DataFrame to avoid modifying the original 'df' in place for other analyses.
df_troubleshoot = df.copy()

df_troubleshoot.loc[0, 'network_latency'] = 1000  # Simulate high network latency
df_troubleshoot.loc[0, 'cpu_usage'] = 85           # Also simulate high CPU usage to diversify anomalies

# Re-run anomaly detection on the modified troubleshooting data.
# IMPORTANT: Pass only numerical columns to the detect_anomalies function.
numeric_data_troubleshoot = df_troubleshoot.select_dtypes(include=[float, int])
df_troubleshoot['anomaly'] = detect_anomalies(numeric_data_troubleshoot)

# Re-calculate z-scores for the troubleshooting scenario based on the modified data.
# This helps in pinpointing which specific metrics are contributing to the anomaly in this scenario.
from scipy.stats import zscore # Ensure zscore is available
z_scores_troubleshoot = numeric_data_troubleshoot.apply(zscore)

# Function to identify anomalous columns for each row, now correctly using troubleshooting z-scores and columns.
# Note: The original find_anomalous_columns function used global 'numeric_data' and 'z_scores'.
# To ensure correctness for the troubleshooting scenario, we inline the logic or pass the correct context.
df_troubleshoot['anomalous_columns'] = df_troubleshoot.apply(
    lambda row: [col for col in numeric_data_troubleshoot.columns if abs(z_scores_troubleshoot.loc[row.name, col]) > 3]
    if row['anomaly'] == -1 else [], axis=1
)

# Prepare the test data (X_test) for root cause analysis from the modified troubleshooting DataFrame.
# Exclude non-numeric columns and the target variable 'anomaly'.
X_test_rca = df_troubleshoot.drop(columns=['timestamp', 'anomaly', 'anomalous_columns'], errors='ignore')

# Run the root cause analysis using the previously trained model.
# This predicts if the rows in the troubleshooting data are anomalous or normal.
predicted_anomalies_test = root_cause_analysis(X_train_rca, y_train_rca, X_test_rca)

# Find the first anomalous entry in the troubleshooting scenario for detailed analysis.
anomalous_entry = df_troubleshoot[df_troubleshoot['anomaly'] == -1].iloc[0] if (df_troubleshoot['anomaly'] == -1).any() else None

if anomalous_entry is not None:
    detected_columns = anomalous_entry['anomalous_columns']
    print(f"Detected anomaly at timestamp: {anomalous_entry['timestamp']}")
    print(f"Contributing anomalous columns: {detected_columns}")

    # Map detected anomalous columns to a specific root cause string for solution recommendation.
    # This mapping can be expanded for more granular root causes to provide more specific advice.
    root_cause_str = "unknown_issue"
    if "network_latency" in detected_columns: # Check if network_latency was identified as anomalous
        root_cause_str = "network_error"
    elif "cpu_usage" in detected_columns: # Check if cpu_usage was identified as anomalous
        root_cause_str = "high_cpu_usage"
    elif "disk_io" in detected_columns:
        root_cause_str = "disk_issue"
    elif "memory_usage" in detected_columns:
        root_cause_str = "high_memory_usage"
    elif "error_rate" in detected_columns:
        root_cause_str = "application_error"
    # If multiple columns are anomalous, you might implement a more complex logic to prioritize or combine causes.

    solution = recommend_solution(root_cause_str)
    print(f"Recommended solution: {solution}")
else:
    print("No anomalies detected in the troubleshooting scenario.")

Detected anomaly at timestamp: 2024-01-01 00:00:00
Contributing anomalous columns: ['cpu_usage', 'network_latency']
Recommended solution: Restart the network service, check network configuration, or contact network team.
